# Homework: Local Search

Students implement Hill-Climbing Search, Local Beam Search, and Simulated Annealing Search algorithms following TODO 1 - 3. \
Students can add supporting attributes and methods to the three classes as needed.

# Libraries

In [1]:
import os
import heapq
import math
import random

# Graph class

In [2]:
# Directed, weighted graphs
class Graph:
  def __init__(self):
    self.AL = dict() # adjacency list
    self.V = 0
    self.E = 0
    self.H = dict()

  def __str__(self):
    res = 'V: %d, E: %d\n'%(self.V, self.E)
    for u, neighbors in self.AL.items():
      line = '%d: %s\n'%(u, str(neighbors))
      res += line
    for u, h in self.H.items():
      line = 'h(%d) = %d\n'%(u, h)
    return res

  def print(self):
    print(str(self))

  def load_from_file(self, filename):
    '''
        Example input file:
            V E
            u v w
            u v w
            u v w
            ...
            u1 h1
            u2 h2
            u3 h3
            ...

        # input.txt
        7 8
        0 1 5
        0 2 6
        1 3 12
        1 4 9
        2 5 5
        3 5 8
        3 6 7
        4 6 4
        0 14
        1 13
        2 12
        3 11
        4 10
        5 9
        6 0
    '''
    if os.path.exists(filename):
      with open(filename) as g:
        self.V, self.E = [int(it) for it in g.readline().split()]
        for i in range(self.E):
          line = g.readline()
          u, v, w = [int(it) for it in line.strip().split()]
          if u not in self.AL:
            self.AL[u] = []
          self.AL[u].append((v, w))
        for i in range(self.V):
          line = g.readline()
          u, h = [int(it) for it in line.strip().split()]
          self.H[u] = h

In [3]:
g = Graph()
g.load_from_file('input.txt')
g.print()

V: 7, E: 8
0: [(1, 5), (2, 6)]
1: [(3, 12), (4, 9)]
2: [(5, 5)]
3: [(5, 8), (6, 7)]
4: [(6, 4)]



# Search Strategies

In [4]:
class LocalSearchStrategy:
  def search(self, g: Graph, src: int) -> tuple:
    '''
    return a tuple (u, p) in which
      u: the local maximum state
      p: the priority/weight/desirability/cost of u
    '''
    return (None, None)

In [5]:
class HillClimbingSearch(LocalSearchStrategy):
  def search(self, g: Graph, src: int) -> tuple:
    '''
    return a tuple (u, p) in which
      u: the local maximum state
      p: the priority/weight/desirability/cost of u

    Note: weight of a node u = path_cost to u + heuristic value of u (similar to A*)
    '''
    current_u = src
    current_g = 0
    current_cost = current_g + g.H.get(current_u, 0)

    while True:
      neighbors = g.AL.get(current_u, [])
      if not neighbors:
        return (current_u, current_cost)

      best_neighbor = None
      best_cost = float('inf')
      best_g = 0

      for v, w in neighbors:
        v_g = current_g + w
        v_cost = v_g + g.H.get(v, 0)
        
        if v_cost < best_cost:
          best_cost = v_cost
          best_neighbor = v
          best_g = v_g

      if best_cost >= current_cost:
        return (current_u, current_cost)

      current_u = best_neighbor
      current_g = best_g
      current_cost = best_cost

In [6]:
class LocalBeamSearch(LocalSearchStrategy):
  def __init__(self, n=2):
    self.n = n
    
    
  def search(self, g: Graph, src: int) -> tuple:
    '''
    return a tuple (u, p) in which
      u: the local maximum state
      p: the priority/weight/desirability/cost of u

    Note:
    - weight of a node u = path_cost to u + heuristic value of u (similar to A*)
    - parameter n is provided in the constructor
    '''
    initial_cost = 0 + g.H.get(src, 0)
    beam = [(initial_cost, 0, src)]
    best_overall = (initial_cost, src)

    while True:
      next_states = []
      for cost, current_g, u in beam:
        for v, w in g.AL.get(u, []):
          v_g = current_g + w
          v_cost = v_g + g.H.get(v, 0)
          next_states.append((v_cost, v_g, v))

      if not next_states:
        return (best_overall[1], best_overall[0])

      next_states.sort(key=lambda x: x[0])
      best_next = next_states[0]

      if best_next[0] >= best_overall[0]:
        return (best_overall[1], best_overall[0])

      best_overall = (best_next[0], best_next[2])
      beam = next_states[:self.n]

In [ ]:
class SimulatedAnnealingSearch(LocalSearchStrategy):
  def search(self, g: Graph, src: int) -> tuple:
    '''
    return a tuple (u, p) in which
      u: the local maximum state
      p: the priority/weight/desirability/cost of u

    Note: schedule(t) = 1/(t^2) with t is the iteration step
    '''
    current_u = src
    current_g = 0
    current_cost = current_g + g.H.get(current_u, 0)

    t = 1
    while t < 10000:
      T = 1.0 / (t ** 2)
      if T < 1e-10:
        break

      neighbors = g.AL.get(current_u, [])
      if not neighbors:
        return (current_u, current_cost)

      v, w = random.choice(neighbors)
      next_g = current_g + w
      next_cost = next_g + g.H.get(v, 0)

      delta_E = current_cost - next_cost 

      if delta_E > 0:
        current_u = v
        current_g = next_g
        current_cost = next_cost
      else:
        probability = math.exp(delta_E / T)
        if random.random() < probability:
          current_u = v
          current_g = next_g
          current_cost = next_cost

      t += 1

    return (current_u, current_cost)

# Evaluation

In [8]:
hcs = HillClimbingSearch()
lbs = LocalBeamSearch()
sas = SimulatedAnnealingSearch()

for stg in [hcs, lbs, sas]:
  print(stg)
  u, p = stg.search(g, 0)
  print(u, p)

0 14
0 14
0 14


# Submission

*   Students download the notebook after completion
*   Rename the notebook in which inserting your student ID at the beginning. \
For example, **123456-lec06-LocalSearch-HW.ipynb**
*   Finally, submit the file